# PlantPepDB

In [ ]:
!pip install requests pandas beautifulsoup4 tqdm lxml

In [ ]:
# CÉLULA 1 - SETUP E FUNÇÃO DE EXTRAÇÃO
import requests
import pandas as pd
import io
import time
import os
import zipfile
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor, as_completed

In [ ]:
def baixar_peptideo_como_df(pep_id):
    url_pagina = f"http://14.139.61.8/PlantPepDB/pages/information.php?id={pep_id}"
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

    try:
        # 1. Acessa a página
        res_pagina = requests.get(url_pagina, headers=headers, timeout=15)
        res_pagina.raise_for_status()

        # 2. Encontra o link de download
        sopa = BeautifulSoup(res_pagina.text, 'html.parser')
        botao_download = sopa.find('a', href=lambda href: href and "download_info.php" in href)

        if botao_download:
            url_download = f"http://14.139.61.8/PlantPepDB/pages/{botao_download['href']}"

            # 3. Baixa o conteúdo do arquivo
            res_arquivo = requests.get(url_download, headers=headers, timeout=15)
            res_arquivo.raise_for_status()

            # 4. A MÁGICA: Transforma o texto bruto direto em um DataFrame do Pandas!
            df = pd.read_csv(io.StringIO(res_arquivo.text), sep='\t')
            return df

        else:
            # Silenciado o print de erro individual para não poluir o terminal durante milhares de requisições
            pass

    except Exception as e:
        pass

    # Retorna DataFrame vazio se algo der errado
    return pd.DataFrame()

In [ ]:
# Criamos a lista de todos os IDs possíveis, do 1 ao 6172
lista_ids = [f"PPepDB_{i}" for i in range(1, 6173)]

print(f"Iniciando a extração para {len(lista_ids)} peptídeos...")
tempo_inicio = time.time()

dataframes_individuais = []
erros_ou_vazios = 0

# Usamos 15 workers para acelerar o processo sem derrubar o servidor deles
with ThreadPoolExecutor(max_workers=15) as executor:
    # Submete todas as tarefas
    futuros = {executor.submit(baixar_peptideo_como_df, pep_id): pep_id for pep_id in lista_ids}

    processados = 0
    for futuro in as_completed(futuros):
        df_resultado = futuro.result()

        # Se o DF veio preenchido, adicionamos à lista
        if df_resultado is not None and not df_resultado.empty:
            dataframes_individuais.append(df_resultado)
        else:
            erros_ou_vazios += 1

        processados += 1

        # Printa o progresso a cada 200 peptídeos concluídos
        if processados % 200 == 0:
            print(f"Progresso: {processados} / {len(lista_ids)} concluídos...")

tempo_total = time.time() - tempo_inicio
print(f"\nExtração finalizada em {tempo_total:.2f} segundos!")
print(f"Sucessos: {len(dataframes_individuais)} | Falhas/Vazios: {erros_ou_vazios}")

Iniciando a extração para 6172 peptídeos...
Progresso: 200 / 6172 concluídos...
Progresso: 400 / 6172 concluídos...
Progresso: 600 / 6172 concluídos...
Progresso: 800 / 6172 concluídos...
Progresso: 1000 / 6172 concluídos...
Progresso: 1200 / 6172 concluídos...
Progresso: 1400 / 6172 concluídos...
Progresso: 1600 / 6172 concluídos...
Progresso: 1800 / 6172 concluídos...
Progresso: 2000 / 6172 concluídos...
Progresso: 2200 / 6172 concluídos...
Progresso: 2400 / 6172 concluídos...
Progresso: 2600 / 6172 concluídos...
Progresso: 2800 / 6172 concluídos...
Progresso: 3000 / 6172 concluídos...
Progresso: 3200 / 6172 concluídos...
Progresso: 3400 / 6172 concluídos...
Progresso: 3600 / 6172 concluídos...
Progresso: 3800 / 6172 concluídos...
Progresso: 4000 / 6172 concluídos...
Progresso: 4200 / 6172 concluídos...
Progresso: 4400 / 6172 concluídos...
Progresso: 4600 / 6172 concluídos...
Progresso: 4800 / 6172 concluídos...
Progresso: 5000 / 6172 concluídos...
Progresso: 5200 / 6172 concluídos..

In [ ]:
print("Concatenando todos os dados em um único DataFrame Gigante...")

if dataframes_individuais:
    df_completo = pd.concat(dataframes_individuais, ignore_index=True)

    print("\n" + "="*50)
    print("EXTRAÇÃO BEM SUCEDIDA!")
    print(f"TAMANHO DO DATAFRAME FINAL: {df_completo.shape}")
    print("="*50 + "\n")

    # Exibe as 5 primeiras linhas para você validar
    display(df_completo.head())
else:
    print("[Aviso] Nenhum dado foi extraído. Verifique a conexão com o servidor.")

Concatenando todos os dados em um único DataFrame Gigante...

EXTRAÇÃO BEM SUCEDIDA!
TAMANHO DO DATAFRAME FINAL: (6172, 17)



,PPepDB-ID,Peptide Name,PMID,Plant source,Plant Family,Peptide Family,Peptide function,Peptide function description,Activity against,Sequence,Sequence Length,Validation,Avg. Molecular wt(Da),Monoisotopic molecular wt(Da),pI,Method,Additional information
0,PPepDB_11,--NA--,doi.org/10.5772/54565,Linum usitatissimum,Linaceae,--NA--,Antihypertensive,--NA--,--NA--,ARDLPGQ,7,Experimental evidence at protein level,755.83,755.39,5.88,EDUF,assay: Udenigwe et al.
1,PPepDB_15,--NA--,8829536,Zea mays,Poaceae,--NA--,Antihypertensive,--NA--,--NA--,ASY,3,Experimental evidence at protein level,339.35,339.14,5.57,RP-HPLC,assay: Cushman and Cheung (1971)
2,PPepDB_4,--NA--,"23271625, 23598136",Chlorella vulgaris,Chlorellaceae,--NA--,Antihypertensive,--NA--,--NA--,AFL,3,Experimental evidence at protein level,349.43,349.20,5.57,"IEC, GFC, HPLC",assay: Cushman and Cheung (1971)
3,PPepDB_2,--NA--,--NA--,Glycine max,Fabaceae,--NA--,Antihypertensive,--NA--,--NA--,AEFGS,5,Experimental evidence at protein level,509.52,509.21,4.00,--NA--,assay: ND
4,PPepDB_9,--NA--,19443002,Amaranthus hypochondriacus,Amaranthaceae,--NA--,Antihypertensive,--NA--,--NA--,ALEP,4,Experimental evidence at protein level,428.49,428.23,4.00,--NA--,assay: ND


In [ ]:
# CONVERTER df_completo → df_tratado (formato importador)
def limpar(v):
    if pd.isna(v):
        return None
    v = str(v).strip()
    if v == "" or v.upper() == "NA" or v == "--NA--":
        return None
    return v

def preencher(v):
    v = limpar(v)
    if v is None:
        return "Not found"
    return v

def tratar_validation(v):
    v = preencher(v).lower()
    if "experimental" in v:
        return "Experimentally Validated"
    if "predict" in v or "homology" in v:
        return "Predicted"
    return "Not found"

def tratar_activity(v):
    txt = preencher(v)
    if txt == "Not found":
        return txt

    txt = txt.lower()
    atividades = []

    mapa = {
        "antibacterial": "Antibacterial",
        "antifungal": "Antifungal",
        "antiparasitic": "Antiparasitic",
        "antitumour": "Antitumour",
        "anticancer": "Antitumour",
        "antiviral": "Antiviral",
        "insect": "Insecticida"
    }

    for chave, valor in mapa.items():
        if chave in txt:
            atividades.append(valor)

    if len(atividades) == 0:
        return "Not found"

    return ", ".join(sorted(set(atividades)))

def tratar_pubmed(v):
    v = limpar(v)
    if v is None:
        return "Not Available"
    return v

df_tratado = pd.DataFrame()

df_tratado["name"] = df_completo["Peptide Name"].apply(preencher)
df_tratado["sequence"] = df_completo["Sequence"].apply(preencher)
df_tratado["organism"] = df_completo["Plant source"].apply(preencher)
df_tratado["activity"] = df_completo["Peptide function"].apply(tratar_activity)
df_tratado["validation"] = df_completo["Validation"].apply(tratar_validation)

# opcionais
df_tratado["uniprot"] = "Not found"
df_tratado["pdb"] = "Not found"

# associação reference/pubmed
df_tratado["pubmed"] = df_completo["PMID"].apply(tratar_pubmed)
df_tratado["reference"] = "Not found"

# sequência obrigatória
df_tratado = df_tratado[df_tratado["sequence"] != "Not found"]

# sequência única
df_tratado = df_tratado.drop_duplicates(subset=["sequence"])

# remover nomes vazios
df_tratado = df_tratado[df_tratado["name"] != "Not found"]

# reset índice
df_tratado = df_tratado.reset_index(drop=True)

print("Linhas finais:", len(df_tratado))
display(df_tratado.head())

Linhas finais: 5624


,name,sequence,organism,activity,validation,uniprot,pdb,pubmed,reference
0,anticancer peptide from algae protein waste (C...,VECYGPNRPQF,"Chlorella vulgaris, Oryza sativa",Antitumour,Experimentally Validated,Not found,Not found,"23271625, 19916544",Not found
1,ACE inhibitor from wakame,KFYG,Undaria pinnatifida,Not found,Experimentally Validated,Not found,Not found,"11091100, 23271625, 23598136, 24915368, 978082...",Not found
2,ACE inhibitor from wakame,YKYY,Undaria pinnatifida,Not found,Experimentally Validated,Not found,Not found,"11091100, 23271625, 23598136, 24915368, 978082...",Not found
3,Angiotensin I-converting enzyme inhibitor 3,AY,"Fagopyrum tataricum, Zea mays, Triticum aestiv...",Not found,Experimentally Validated,Not found,Not found,"17715987, 18211015, 19994857, 23598136, 249153...",Not found
4,Angiotensin I-converting enzyme inhibitor 4,AF,"Triticum aestivum, Glycine max, Theobroma cacao",Not found,Experimentally Validated,Not found,Not found,"19994857, 23598136, 24915368, 8829536, 10442764",Not found


In [ ]:
# SALVAR CSV E COMPACTAR
import zipfile
import os

ARQUIVO_CSV_IMPORT = "PlantPepDB_import.csv"
ARQUIVO_ZIP = "PlantPepDB_import.zip"

print(f"Salvando o CSV local como: {ARQUIVO_CSV_IMPORT}")
df_tratado.to_csv(ARQUIVO_CSV_IMPORT, index=False, encoding="utf-8")

print(f"Compactando em: {ARQUIVO_ZIP}")
with zipfile.ZipFile(ARQUIVO_ZIP, 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(ARQUIVO_CSV_IMPORT)

# Opcional: remover o CSV solto para deixar o diretório limpo
# os.remove(ARQUIVO_CSV_IMPORT)

print("Processo 100% concluído! Arquivo ZIP gerado para importação.")

Salvando o CSV local como: PlantPepDB_import.csv
Compactando em: PlantPepDB_import.zip
Processo 100% concluído! Arquivo ZIP gerado para importação.


# dbAMP

In [ ]:
import requests
import urllib3
import ssl
from requests.adapters import HTTPAdapter
from urllib3.util.ssl_ import create_urllib3_context

# Desativa alertas visuais de SSL no console
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# O adaptador com a correção para o erro de CERT_NONE
class LegacySSLAdapter(HTTPAdapter):
    def init_poolmanager(self, *args, **kwargs):
        ctx = create_urllib3_context()
        ctx.set_ciphers('DEFAULT@SECLEVEL=1')

        # --- A CORREÇÃO ESTÁ AQUI ---
        # Forçamos o contexto a não checar o hostname e ignorar o certificado
        ctx.check_hostname = False
        ctx.verify_mode = ssl.CERT_NONE
        # ----------------------------

        kwargs['ssl_context'] = ctx
        return super(LegacySSLAdapter, self).init_poolmanager(*args, **kwargs)

def baixar_response_txt(pep_id):
    url = f"https://ycclab.cuhk.edu.cn/dbAMP/information.php?db={pep_id}"
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
    nome_arquivo = f"{pep_id}_response.txt"

    print(f"Acessando o endpoint para: {pep_id}...")

    # Criamos a sessão com o adaptador de segurança reduzida
    session = requests.Session()
    session.mount('https://', LegacySSLAdapter())

    try:
        # verify=False agora vai funcionar em harmonia com o adaptador
        res = session.get(url, headers=headers, timeout=20, verify=False)

        print(f"Status Code retornado: {res.status_code}")

        if res.status_code == 200:
            # Salva a resposta bruta em um arquivo de texto
            with open(nome_arquivo, "w", encoding="utf-8") as f:
                f.write(res.text)

            print(f"Sucesso! Todo o conteúdo foi salvo no arquivo: {nome_arquivo}")
        else:
            print(f"Falha na requisição. O servidor retornou: {res.status_code}")

    except Exception as e:
        print(f"Erro ao tentar baixar os dados: {e}")

# Executando para o primeiro peptídeo
baixar_response_txt("dbAMP_00001")

Acessando o endpoint para: dbAMP_00001...
Status Code retornado: 200
Sucesso! Todo o conteúdo foi salvo no arquivo: dbAMP_00001_response.txt


In [ ]:
import pandas as pd
from bs4 import BeautifulSoup

# Lendo o arquivo HTML salvo localmente
with open("dbAMP_00001_response.txt", "r", encoding="utf-8") as f:
    html_content = f.read()

sopa = BeautifulSoup(html_content, 'html.parser')

# Isolando a aba 'home' onde fica a tabela principal
aba_home = sopa.find('div', id='home')

if aba_home:
    tabela = aba_home.find('table', class_='table')

    # Dicionário para armazenar a linha
    dados = {"dbAMP ID": "dbAMP_00001"}

    # Pegando apenas as linhas dentro do tbody da tabela correta
    linhas = tabela.find('tbody').find_all('tr')

    for linha in linhas:
        colunas = linha.find_all('td')

        # Só processa se a linha tiver exatamente 2 colunas (Chave e Valor)
        if len(colunas) == 2:
            chave = colunas[0].get_text(strip=True)

            # O separator=", " troca os <br> e <span> por vírgulas para não colar as palavras
            valor = colunas[1].get_text(separator=", ", strip=True)

            # Limpeza final de vírgulas que possam sobrar nas bordas
            valor = valor.strip(", ")

            dados[chave] = valor

    # Transformando em DataFrame
    df_dbamp_00001 = pd.DataFrame([dados])

    print("\n" + "="*50)
    print(f"TAMANHO DO DATAFRAME: {df_dbamp_00001.shape}")
    print("="*50 + "\n")

    # Exibe no Notebook
    display(df_dbamp_00001)

else:
    print("Aba de informações básicas não encontrada no HTML.")


TAMANHO DO DATAFRAME: (1, 9)



,dbAMP ID,Name,Source,Length,Activity,Sequence,Taxonomy,Experimental Evidence,PubMed
0,dbAMP_00001,"Synthetic construct, 4A",Synthetic construct,43,"Antibacterial, Antimicrobial",AAAAGSVWGAVNYTSDCNGECKRRGYKGGYCGSFANVNCWCET,"Eukaryota,Metazoa,Arthropoda,Insecta,Lepidopte...",NO,11580275


In [ ]:
# CÉLULA 1 - IMPORTAÇÕES E ADAPTADOR DE SEGURANÇA
import requests
import urllib3
import ssl
import pandas as pd
import time
import zipfile
import os
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from urllib3.util.ssl_ import create_urllib3_context
from concurrent.futures import ThreadPoolExecutor, as_completed

In [ ]:
# Desativa alertas visuais de SSL no console
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Configurações de exportação
ARQUIVO_CSV_DBAMP = "dbAMP_completo.csv"
ARQUIVO_ZIP_DBAMP = "dbAMP_completo.zip"

# O adaptador com a correção para o erro de handshake e CERT_NONE
class LegacySSLAdapter(HTTPAdapter):
    def init_poolmanager(self, *args, **kwargs):
        ctx = create_urllib3_context()
        ctx.set_ciphers('DEFAULT@SECLEVEL=1')
        ctx.check_hostname = False
        ctx.verify_mode = ssl.CERT_NONE
        kwargs['ssl_context'] = ctx
        return super(LegacySSLAdapter, self).init_poolmanager(*args, **kwargs)


In [ ]:
def extrair_dados_dbamp(numero_id):
    """
    Acessa o endpoint dbAMP, fura o bloqueio SSL, lê o HTML em memória,
    filtra a aba 'home' e extrai a tabela limpa para um dicionário.
    """
    # Formata o número (ex: 1 vira dbAMP_00001)
    pep_id = f"dbAMP_{numero_id:05d}"
    url = f"https://ycclab.cuhk.edu.cn/dbAMP/information.php?db={pep_id}"
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

    # Criamos a sessão segura por requisição
    session = requests.Session()
    session.mount('https://', LegacySSLAdapter())

    try:
        # Bate no servidor
        res = session.get(url, headers=headers, timeout=20, verify=False)

        if res.status_code == 200:
            sopa = BeautifulSoup(res.text, 'html.parser')

            # Procura especificamente a aba de informações básicas
            aba_home = sopa.find('div', id='home')

            if aba_home:
                tabela = aba_home.find('table', class_='table')
                if tabela and tabela.find('tbody'):
                    dados = {"dbAMP ID": pep_id}
                    linhas = tabela.find('tbody').find_all('tr')

                    for linha in linhas:
                        colunas = linha.find_all('td')
                        if len(colunas) == 2:
                            chave = colunas[0].get_text(strip=True)

                            # Separador mágico que resolve as tags <br> e <span>
                            valor = colunas[1].get_text(separator=", ", strip=True)
                            valor = valor.strip(", ")

                            if valor:
                                dados[chave] = valor
                            else:
                                dados[chave] = "Not found"

                    return dados

    except Exception:
        pass # Ignora falhas de timeout para não parar o loop gigante

    return None

In [ ]:
# CÉLULA 3 - MULTITHREADING (Pode demorar entre 15 a 40 minutos)
TOTAL_REGISTROS = 35600
lista_numeros = list(range(1, TOTAL_REGISTROS + 1))

print(f"Iniciando a extração massiva para {TOTAL_REGISTROS} registros...")
tempo_inicio = time.time()

resultados_extraidos = []
falhas_ou_vazios = 0

# Limitado a 15 workers para balancear velocidade sem derrubar o servidor chinês
with ThreadPoolExecutor(max_workers=15) as executor:
    futuros = {executor.submit(extrair_dados_dbamp, num): num for num in lista_numeros}

    processados = 0
    for futuro in as_completed(futuros):
        resultado = futuro.result()

        # Só adiciona na base se a tabela foi encontrada e preenchida
        if resultado and len(resultado) > 1:
            resultados_extraidos.append(resultado)
        else:
            falhas_ou_vazios += 1

        processados += 1
        if processados % 1000 == 0:
            print(f"Progresso: {processados} / {TOTAL_REGISTROS} concluídos...")

tempo_total = time.time() - tempo_inicio
print(f"\nExtração 100% concluída em {tempo_total:.2f} segundos!")
print(f"Peptídeos válidos extraídos: {len(resultados_extraidos)} | Falhas/Inexistentes: {falhas_ou_vazios}")

Iniciando a extração massiva para 35600 registros...
Progresso: 1000 / 35600 concluídos...
Progresso: 2000 / 35600 concluídos...
Progresso: 3000 / 35600 concluídos...
Progresso: 4000 / 35600 concluídos...
Progresso: 5000 / 35600 concluídos...
Progresso: 6000 / 35600 concluídos...
Progresso: 7000 / 35600 concluídos...
Progresso: 8000 / 35600 concluídos...
Progresso: 9000 / 35600 concluídos...
Progresso: 10000 / 35600 concluídos...
Progresso: 11000 / 35600 concluídos...
Progresso: 12000 / 35600 concluídos...
Progresso: 13000 / 35600 concluídos...
Progresso: 14000 / 35600 concluídos...
Progresso: 15000 / 35600 concluídos...
Progresso: 16000 / 35600 concluídos...
Progresso: 17000 / 35600 concluídos...
Progresso: 18000 / 35600 concluídos...
Progresso: 19000 / 35600 concluídos...
Progresso: 20000 / 35600 concluídos...
Progresso: 21000 / 35600 concluídos...
Progresso: 22000 / 35600 concluídos...
Progresso: 23000 / 35600 concluídos...
Progresso: 24000 / 35600 concluídos...
Progresso: 25000 / 3

In [ ]:
print("Organizando os dados em formato tabular...")

if resultados_extraidos:
    # O Pandas transforma a lista de dicionários automaticamente em colunas e linhas
    df_completo = pd.DataFrame(resultados_extraidos)

    # Preenche qualquer buraco que tenha ficado nulo
    df_completo.fillna("Not found", inplace=True)

    print("\n" + "="*50)
    print("RESUMO DA BASE DE DADOS (dbAMP):")
    print(f"Total de Linhas: {df_completo.shape[0]}")
    print(f"Total de Colunas: {df_completo.shape[1]}")
    print("="*50 + "\n")

    print("Colunas encontradas e alinhadas:")
    print(df_completo.columns.tolist())

    # Mostra uma prévia dos dados
    display(df_completo.head())
else:
    print("Falha Crítica: Nenhum dado foi salvo. Verifique sua conexão com a rede.")

Organizando os dados em formato tabular...

RESUMO DA BASE DE DADOS (dbAMP):
Total de Linhas: 35578
Total de Colunas: 12

Colunas encontradas e alinhadas:
['dbAMP ID', 'Length', 'Activity', 'Sequence', 'Experimental Evidence', 'PubMed', 'Name', 'Source', 'Taxonomy', 'UniprotKB ID/AC', 'Target', 'Patent ID']


,dbAMP ID,Length,Activity,Sequence,Experimental Evidence,PubMed,Name,Source,Taxonomy,UniprotKB ID/AC,Target,Patent ID
0,dbAMP_00003,18,AntiHIV,AAAPAATLEEHMTACQGV,NO,8207412,Not found,Not found,Not found,Not found,Not found,Not found
1,dbAMP_00002,18,AntiHIV,AAANPGLLETSEGCRQIL,NO,8207412,Not found,Not found,Not found,Not found,Not found,Not found
2,dbAMP_00006,60,"Antibacterial, AntiGram +, Antimicrobial, Chem...",AAEFPDFYDSEEQMGPHQEAEDEKDRADQRVLTEEEKKELENLAAM...,NO,"18181560, 19341344","Chrombacin, Bos taurus (Bovine), Bos taurus","Bos taurus (Bovine), Bos taurus","Eukaryota,Metazoa,Chordata,Mammalia,Rodentia,M...",Not found,Not found,Not found
3,dbAMP_00005,40,"Antibacterial, AntiGram +, AntiGram&nbsp-, Ant...",AACSDRAHGHICESFKSFCKDSGRNGVKLRANCKKTCGLC,NO,"16890198, 16890198","jellyfish, Aurelia aurita, Aurelin (jellyfish,...","jellyfish, Aurelia aurita, Aurelia aurita (Moo...","Eukaryota,Metazoa,Cnidaria,Scyphozoa,Semaeosto...",AURE_AURAU (Q0MWV8),"Escherichia coli ML-35p (MIC=7.66μg/ml), Liste...",Not found
4,dbAMP_00004,97,"Antifungal, Antihypertensive",AACARFIDDFCDTLTPNIYRPRDNGQRCYAVNGHRCDFTVFNTNNG...,NO,Not found,Not found,Not found,"Eukaryota,Fungi,Dikarya,Basidiomycota,Agaricom...",APY3_PLECI (P83477),Not found,Not found


In [ ]:
# CONVERTER dbAMP → FORMATO DO BANCO
import pandas as pd

# --------------------------
# limpeza e preenchimento
# --------------------------
def limpar(v):
    if pd.isna(v):
        return None
    v = str(v).strip()
    if v == "" or v == "--NA--" or v.upper() == "NA" or v == "Not found":
        return None
    return v

def preencher(v):
    v = limpar(v)
    if v is None:
        return "Not found"
    return v

# --------------------------
# organismo
# usa Source, senão última taxonomia
# --------------------------
def pegar_organismo(row):
    src = limpar(row.get("Source"))
    if src:
        return src

    tax = limpar(row.get("Taxonomy"))
    if tax:
        return tax.split(",")[-1].strip()

    return "Not found"

# --------------------------
# nome
# --------------------------
def pegar_nome(row):
    nome = limpar(row.get("Name"))
    if nome:
        return nome
    return row.get("dbAMP ID", "Not found")

# --------------------------
# atividade
# --------------------------
MAPA = {
    "antibacterial": "Antibacterial",
    "antifungal": "Antifungal",
    "antiparasitic": "Antiparasitic",
    "antiviral": "Antiviral",
    "antitumour": "Antitumour",
    "anticancer": "Antitumour",
    "antimicrobial": "Antibacterial",
    "insecticidal": "Insecticida"
}

def atividade(txt):
    txt = str(preencher(txt)).lower()
    out = []

    for k in MAPA:
        if k in txt:
            out.append(MAPA[k])

    if len(out) == 0:
        return "Not found"

    return ", ".join(sorted(set(out)))

# --------------------------
# validação
# --------------------------
def valid(v):
    v = str(preencher(v)).upper()
    if v == "YES":
        return "Experimentally Validated"
    return "Predicted"

# ==================================================
# montar dataframe
# ==================================================
df_tratado = pd.DataFrame()

df_tratado["name"] = df_completo.apply(pegar_nome, axis=1)
df_tratado["sequence"] = df_completo.get("Sequence", pd.Series(dtype=str)).apply(preencher)
df_tratado["organism"] = df_completo.apply(pegar_organismo, axis=1)
df_tratado["activity"] = df_completo.get("Activity", pd.Series(dtype=str)).apply(atividade)
df_tratado["validation"] = df_completo.get("Experimental Evidence", pd.Series(dtype=str)).apply(valid)
df_tratado["uniprot"] = df_completo.get("UniprotKB ID/AC", pd.Series(dtype=str)).apply(preencher)
df_tratado["pdb"] = "Not found"
df_tratado["pubmed"] = df_completo.get("PubMed", pd.Series(dtype=str)).apply(preencher)
df_tratado["reference"] = "Not found"

# --------------------------
# filtros finais
# --------------------------
# remover sequência inválida
df_tratado = df_tratado.query('sequence != "Not found"')

# sequência única
df_tratado = df_tratado.drop_duplicates(subset=["sequence"])

# resetar índices
df_tratado = df_tratado.reset_index(drop=True)

print("Tamanho do DataFrame final:", df_tratado.shape)
display(df_tratado.head())

Tamanho do DataFrame final: (35374, 9)


,name,sequence,organism,activity,validation,uniprot,pdb,pubmed,reference
0,dbAMP_00003,AAAPAATLEEHMTACQGV,Not found,Not found,Predicted,Not found,Not found,8207412,Not found
1,dbAMP_00002,AAANPGLLETSEGCRQIL,Not found,Not found,Predicted,Not found,Not found,8207412,Not found
2,"Chrombacin, Bos taurus (Bovine), Bos taurus",AAEFPDFYDSEEQMGPHQEAEDEKDRADQRVLTEEEKKELENLAAM...,"Bos taurus (Bovine), Bos taurus",Antibacterial,Predicted,Not found,Not found,"18181560, 19341344",Not found
3,"jellyfish, Aurelia aurita, Aurelin (jellyfish,...",AACSDRAHGHICESFKSFCKDSGRNGVKLRANCKKTCGLC,"jellyfish, Aurelia aurita, Aurelia aurita (Moo...",Antibacterial,Predicted,AURE_AURAU (Q0MWV8),Not found,"16890198, 16890198",Not found
4,dbAMP_00004,AACARFIDDFCDTLTPNIYRPRDNGQRCYAVNGHRCDFTVFNTNNG...,Coprinus comatus,Antifungal,Predicted,APY3_PLECI (P83477),Not found,Not found,Not found


# APD6 Antimicrobial Peptide Database

In [29]:
import requests
import re
from bs4 import BeautifulSoup
import pandas as pd
import time

SEARCH_URL = "https://aps.unmc.edu/database/result"
DETAIL_URL = "https://aps.unmc.edu/database/peptide"
OUTPUT_FILE = "banco_peptideos_organizado.csv"

search_payload = {
    'ID': '', 'Name': 'plants', 'Name2': '', 'Name3': '', 'source': '',
    'Sequence': '', 'Sequence2': '', 'Length': 'Any', 'Netcharge': 'Any',
    'HydrophobicPer': 'Any', 'Location': 'Any', 'LocationID': '',
    'Type': 'Any', 'Method': 'Any', 'info': '', 'info2': '', 'info3': '',
    'author': '', 'Sort': 'ID'
}

In [30]:
print("1/4 -> Disparando busca geral para extrair os IDs do APD6...")
try:
    response = requests.post(SEARCH_URL, data=search_payload, timeout=30)
    if response.status_code == 200:
        padrao_id = r'name="peptides\[\]"\s+value="([A-Za-z0-9]+)"'
        ids_peptideos = re.findall(padrao_id, response.text)
        ids_unicos = list(dict.fromkeys(ids_peptideos))
        print(f"Sucesso! Encontrados {len(ids_unicos)} IDs únicos para processar.\n")
    else:
        print(f"Erro no Passo 1. Status Code: {response.status_code}")
        ids_unicos = []
except Exception as e:
    print(f"Falha de conexão no Passo 1: {e}")
    ids_unicos = []

1/4 -> Disparando busca geral para extrair os IDs do APD6...
Sucesso! Encontrados 394 IDs únicos para processar.



In [31]:
peptideos_completos = []

if ids_unicos:
    print("2/4 -> Iniciando a extração detalhada de cada peptídeo...")

    for idx, peptide_id in enumerate(ids_unicos, start=1):
        detail_payload = {'ID': peptide_id}
        try:
            res_detail = requests.post(DETAIL_URL, data=detail_payload, timeout=30)
            if res_detail.status_code == 200:
                soup = BeautifulSoup(res_detail.text, "html.parser")
                dados_peptideo = {}
                tabela = soup.find("table", class_="peptide")

                if tabela:
                    linhas = tabela.find_all("tr")
                    for linha in linhas:
                        colunas = linha.find_all("td")
                        if len(colunas) == 2:
                            chave = colunas[0].get_text(strip=True).replace(":", "")

                            # CRÍTICO: Se for Reference ou Additional Info, extraímos o texto + URLs href ocultas
                            if chave in ["Reference", "Additional info"]:
                                texto_puro = colunas[1].get_text(" ", strip=True)
                                links_ocultos = [a['href'] for a in colunas[1].find_all('a', href=True)]
                                valor = texto_puro + " " + " ".join(links_ocultos)
                            else:
                                valor = colunas[1].get_text(" ", strip=True)

                            dados_peptideo[chave] = valor

                    peptideos_completos.append(dados_peptideo)

                    nome_peptideo = dados_peptideo.get("Name/Class", "Sem Nome").split("(")[0].strip()
                    print(f"[{idx}/{len(ids_unicos)}] Extraindo com sucesso: ID {peptide_id} | {nome_peptideo}")
                else:
                    print(f"[{idx}/{len(ids_unicos)}] Alerta: Tabela não encontrada para ID {peptide_id}")
            else:
                print(f"[{idx}/{len(ids_unicos)}] Erro ao acessar ID {peptide_id}. Status: {res_detail.status_code}")

        except Exception as e:
            print(f"[{idx}/{len(ids_unicos)}] Falha no ID {peptide_id}: {e}")

        # Pequena pausa de segurança (respeito ao servidor)
        time.sleep(0.3)


2/4 -> Iniciando a extração detalhada de cada peptídeo...
[1/394] Extraindo com sucesso: ID 00003 | Hs-AFP1
[2/394] Extraindo com sucesso: ID 00004 | Ct-AMP1
[3/394] Extraindo com sucesso: ID 00103 | NmDef02
[4/394] Extraindo com sucesso: ID 00184 | MBP-1
[5/394] Extraindo com sucesso: ID 00194 | Ac-AMP2
[6/394] Extraindo com sucesso: ID 00202 | Mj-AMP2
[7/394] Extraindo com sucesso: ID 00203 | Mj-AMP1
[8/394] Extraindo com sucesso: ID 00236 | Pyrularia thionin
[9/394] Extraindo com sucesso: ID 00237 | Phoratoxin
[10/394] Extraindo com sucesso: ID 00270 | Pn-AMP1
[11/394] Extraindo com sucesso: ID 00274 | Circulin A
[12/394] Extraindo com sucesso: ID 00275 | Circulin B
[13/394] Extraindo com sucesso: ID 00282 | Cyclopsychotride A
[14/394] Extraindo com sucesso: ID 00286 | Rs-AFP1
[15/394] Extraindo com sucesso: ID 00287 | Rs-AFP2
[16/394] Extraindo com sucesso: ID 00288 | So-D1
[17/394] Extraindo com sucesso: ID 00289 | So-D3
[18/394] Extraindo com sucesso: ID 00290 | So-D4
[19/394] Ex

In [32]:
if peptideos_completos:
    print("\n3/4 -> Aplicando regras de Organização e Especificação dos Dados...")
    registros_formatados = []

    for p in peptideos_completos:
        item = {}

        # 1. name (Monovalorado, Obrigatório)
        raw_name = p.get("Name/Class", "Not found")
        item["name"] = raw_name.split("(")[0].strip() if raw_name != "Not found" else "Not found"

        # 2. sequence (Monovalorado, Obrigatório)
        item["sequence"] = p.get("Sequence", "Not found").replace(" ", "")

        # 3. organism (Multivalorado, Obrigatório)
        raw_organism = p.get("Source", "Not found")
        if raw_organism != "Not found":
            # Remove partes da planta que antecedem o nome científico
            clean_organism = re.sub(r'^(seeds|leaves|whole plant|bark|roots|sap|latex|shoots),\s*', '', raw_organism, flags=re.IGNORECASE)
            item["organism"] = clean_organism.strip()
        else:
            item["organism"] = "Not found"

        # 4. activity (Multivalorado, Obrigatório)
        raw_activity = p.get("Activity", "Not found")
        if raw_activity != "Not found":
            atividades_encontradas = [act.strip() for act in re.split(r'[,&]', raw_activity)]
            valid_acts = []
            for act in atividades_encontradas:
                if "Gram" in act or "Antibacterial" in act:
                    valid_acts.append("Antibacterial")
                elif "Antifungal" in act:
                    valid_acts.append("Antifungal")
                elif "Antiviral" in act:
                    valid_acts.append("Antiviral")
                elif "Antiparasitic" in act:
                    valid_acts.append("Antiparasitic")
                elif "Antitumor" in act or "Antitumour" in act or "Cancer" in act:
                    valid_acts.append("Antitumour")
                elif "Insect" in act:
                    valid_acts.append("Insecticida")

            valid_acts = list(set(valid_acts))
            item["activity"] = ", ".join(valid_acts) if valid_acts else "Not found"
        else:
            item["activity"] = "Not found"

        # 5. validation (Monovalorado, Obrigatório)
        name_lower = raw_name.lower()
        if "predicted" in name_lower or "machine learning" in name_lower:
            item["validation"] = "Predicted"
        elif raw_name == "Not found":
            item["validation"] = "Not found"
        else:
            item["validation"] = "Experimentally Validated"

        # 6. uniprot (Multivalorado, Não Obrigatório)
        swiss_id = p.get("SwissProt ID", p.get("Reference ID", "Not found"))
        match_uniprot = re.search(r'\b([OPQ][0-9A-Z]{5}|[A-NR-Z][0-9][A-Z0-9]{3}[0-9])\b', swiss_id + " " + raw_name)
        item["uniprot"] = match_uniprot.group(1) if match_uniprot else "Not found"

        # 7. pdb (Multivalorado, Não Obrigatório)
        raw_pdb = p.get("3D Structure", "Not found")
        match_pdb = re.search(r'\b([0-9][A-Za-z0-9]{3})\b', raw_pdb + " " + swiss_id)
        item["pdb"] = match_pdb.group(1).upper() if match_pdb else "Not found"

        # 8. reference e 9. pubmed (Multivalorados Associados - Regra ª com "##")
        raw_ref = p.get("Reference", "Not found")
        raw_add_info = p.get("Additional info", "")
        bloco_texto_total = raw_ref + " " + raw_add_info

        # Captura os IDs numéricos mapeados das URLs injetadas no Passo 2
        pubmed_ids = re.findall(r'nih\.gov/(\d+)', bloco_texto_total, flags=re.IGNORECASE)
        if not pubmed_ids:
            pubmed_ids = re.findall(r'pubmed/(\d+)', bloco_texto_total, flags=re.IGNORECASE)
        if not pubmed_ids:
            pubmed_ids = re.findall(r'PubMed:\s*(\d+)', bloco_texto_total, flags=re.IGNORECASE)

        pubmed_ids = list(dict.fromkeys(pubmed_ids)) # Remove duplicados mantendo a ordem

        if pubmed_ids:
            # Limpa links HTTP residuais do campo de texto da referência
            clean_reference_text = re.sub(r'https?://\S+', '', raw_ref).strip()
            clean_reference_text = clean_reference_text.replace("PubMed .", "").strip()
            ref_text = clean_reference_text if clean_reference_text else "Reference literature source"

            # Garante que o número de referências bate exatamente com o número de IDs PubMed (Regra ª)
            refs_list = [f"{ref_text} (Part {i+1})" for i in range(len(pubmed_ids))] if len(pubmed_ids) > 1 else [ref_text]

            item["reference"] = " ## ".join(refs_list)
            item["pubmed"] = " ## ".join(pubmed_ids)
        else:
            clean_reference_text = re.sub(r'https?://\S+', '', raw_ref).replace("PubMed .", "").strip()
            item["reference"] = clean_reference_text if clean_reference_text else "Not found"
            item["pubmed"] = "Not available"

        registros_formatados.append(item)


3/4 -> Aplicando regras de Organização e Especificação dos Dados...


In [35]:
print("4/4 -> Consolidação e limpeza no DataFrame do Pandas...")

ordem_colunas = ["name", "sequence", "organism", "activity", "validation", "uniprot", "pdb", "reference", "pubmed"]
df_final = pd.DataFrame(registros_formatados, columns=ordem_colunas)

# Aplicação da Regra * (Garantir que a sequência de aminoácidos é única)
total_antes = len(df_final)
df_final = df_final.drop_duplicates(subset=["sequence"], keep="first")
total_depois = len(df_final)

if total_antes != total_depois:
    print(f"-> Nota: Removidos {total_antes - total_depois} registos com sequências duplicadas (*).")

# Exportação Final em conformidade com as especificações
df_final.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

print("\n================--------------------------------------------------------")
print(f"Pipeline finalizado com sucesso! Ficheiro salvo em: '{OUTPUT_FILE}'")
print("================--------------------------------------------------------")

# Exibe as primeiras linhas do resultado para validação visual
print("\n--- PRÉVIA DO DATAFRAME FINAL ---")
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
print(df_final.head(5))


4/4 -> Consolidação e limpeza no DataFrame do Pandas...
-> Nota: Removidos 1 registos com sequências duplicadas (*).

================--------------------------------------------------------
Pipeline finalizado com sucesso! Ficheiro salvo em: 'banco_peptideos_organizado.csv'
================--------------------------------------------------------

--- PRÉVIA DO DATAFRAME FINAL ---
      name                                           sequence                                           organism                   activity                validation    uniprot        pdb                                  reference         pubmed
0  Hs-AFP1  DGVKLCDVPSGTWSGHCGSSSKCSQQCKDREHFAYGGACHYQFPSV...                                 Heuchera sanguinea                 Antifungal  Experimentally Validated  Not found  Not found      FEBS Lett. 1995 Jul 17;368(2):257-62.        7628617
1  Ct-AMP1  NLCERASLTWTGNCGNTGHCDTQCRNWESAKHGACHKRGNWKCFCYFDC  Asian pigeonwings, bluebellvine, blue pea, but...  Antibacter